In [1]:
import numpy as np
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt
import keras
from sklearn.linear_model import LogisticRegression

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from collections import Counter

# Load data
# data_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project\New Data.csv"
# data_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project\Train Data.csv"
data_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project\gse2034_9genes_bone.csv"
data = pd.read_csv(data_path, delimiter=",")
display(data)

# Separate features and labels
X = data.iloc[:, :-1]
y = data.iloc[:, -1]

# Step 1: 70% Train, 30% Temp (CV + Test), with stratification
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# Step 2: Split Temp into CV (10.5%) and Test (19.5%) [ratio = 30:70]
X_cv, X_test, y_cv, y_test = train_test_split(
    X_temp, y_temp, test_size=0.65, random_state=42, stratify=y_temp)

# Confirm original class distribution
print("Original class distribution in training set:", Counter(y_train))

# # Apply SMOTE only to the training data
# smote = SMOTE(random_state=42)
# X_train, y_train = smote.fit_resample(X_train, y_train)

# # Confirm new class distribution
# print("After SMOTE:", Counter(y_train))

# # Final split sizes
# print(f"\nFinal Sizes:")
# print(f"  Train: {len(X_train)} (after SMOTE)")
# print(f"  CV:    {len(X_cv)}")
# print(f"  Test:  {len(X_test)}")

# # Overall label distribution in full data
# print("\nFull label distribution:", np.bincount(y.astype(int)))

,217092_x_at,43544_at,Characteristics..Relapse..Metastasis.Bone.
0,-0.317072,0.165847,1
1,-0.020898,0.897427,1
2,-1.068299,2.281557,0
3,-3.446621,1.950300,1
4,-2.389388,1.599062,1
...,...,...,...
640,1.463694,0.647379,1
641,0.136806,-0.052912,0
642,1.341405,-0.521103,0
643,0.169912,0.029760,0


Original class distribution in training set: Counter({0: 373, 1: 78})


In [ ]:
import pandas as pd
from itertools import combinations
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, recall_score, precision_score,
    f1_score, accuracy_score
)

# Define gene names
gene_names = ['201506_at', '205486_at', '216638_s_at', '221619_s_at', '221672_s_at', '35148_at']

# Convert arrays to DataFrames
X_train = pd.DataFrame(X_train, columns=gene_names)
X_cv = pd.DataFrame(X_cv, columns=gene_names)
X_test = pd.DataFrame(X_test, columns=gene_names)

# Store all results
all_results = []

# Detect classification type
n_classes = len(set(y_train))

# Loop through all feature combinations
for k in range(1, len(gene_names) + 1):
    for comb in combinations(gene_names, k):
        features = list(comb)

        # Select features
        X_train_sel = X_train[features]
        X_cv_sel = X_cv[features]
        X_test_sel = X_test[features]

        # Train model
        model = LogisticRegression(max_iter=1000)
        model.fit(X_train_sel, y_train)

        # Predict
        y_train_pred = model.predict(X_train_sel)
        y_cv_pred = model.predict(X_cv_sel)
        y_test_pred = model.predict(X_test_sel)
        y_test_proba = model.predict_proba(X_test_sel)

        # AUC handling
        try:
            if n_classes > 2:
                auc = roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro')
            else:
                auc = roc_auc_score(y_test, y_test_proba[:, 1])
        except:
            auc = float('-inf')

        # Other metrics
        recall = recall_score(y_test, y_test_pred, average='macro', zero_division=0)
        precision = precision_score(y_test, y_test_pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_test_pred, average='macro', zero_division=0)
        train_acc = accuracy_score(y_train, y_train_pred)
        cv_acc = accuracy_score(y_cv, y_cv_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        all_results.append({
            "genes": ' + '.join(features),
            "auc": auc,
            "recall": recall,
            "precision": precision,
            "f1": f1,
            "train_acc": train_acc,
            "cv_acc": cv_acc,
            "test_acc": test_acc
        })

# Create results DataFrame
results_df = pd.DataFrame(all_results)

# Get best model for each metric
summary_rows = []
metrics = {
    "Highest AUC": "auc",
    "Highest Recall": "recall",
    "Highest Precision": "precision",
    "Highest F1-score": "f1",
    "Highest Train Accuracy": "train_acc",
    "Highest CV Accuracy": "cv_acc",
    "Highest Test Accuracy": "test_acc"
}

for label, metric in metrics.items():
    best_row = results_df.loc[results_df[metric].idxmax()]
    summary_rows.append({
        "Metric": label,
        "Genes": best_row["genes"],
        "AUC": best_row["auc"],
        "Recall": best_row["recall"],
        "Precision": best_row["precision"],
        "F1-score": best_row["f1"],
        "Train Accuracy": best_row["train_acc"],
        "CV Accuracy": best_row["cv_acc"],
        "Test Accuracy": best_row["test_acc"]
    })
    
# Save summary to CSV
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("Logistic Regression.csv", index=False)
print("✅ Summary saved to 'Logistic Regression.csv'")

✅ Summary saved to 'Logistic Regression.csv'
